<!--
File: technical_memory.ipynb
Description: Running log of technical decisions, tradeoffs, and gotchas for the josejorge/josejorge GitHub profile README repo.
Author: Jose-Jorge HERNANDEZ
Company: Parlee Conseiller, Inc.
Date: 2026-09-14
Last edit date: 2026-09-14
Version: 1.0.0
-->

# Technical Memory — josejorge/josejorge

This notebook is the *why* behind technical decisions in this repo — not a changelog (that's `git log`) and not a bug log. Each entry below is dated and explains reasoning, tradeoffs weighed, and gotchas hit.

## 2026-09-14 — Activity Graph section was pointing at a dead service

**Symptom:** the README's "📈 Activity Graph" section never rendered.

**Root cause:** it embedded a live `<img>` from `https://github-readme-activity-graph.vercel.app/graph?...`. That public Vercel deployment has been shut down entirely — it returns HTTP 402 with body `DEPLOYMENT_DISABLED`, not a transient outage. Confirmed by direct `curl` below.

**Decision:** rather than embedding *any* live third-party URL (the same class of risk that caused this), follow the repo's own established pattern — already used for `assets/dashboard.png` (Playwright screenshot of the Grafana dashboard, committed every 6h) and `assets/top-langs.svg` (fetched via a GitHub Action, committed) — of generating the asset server-side in CI and committing a static file, so README always renders something even if the generating service is slow or briefly down.

**Source swap:** `ghchart.rshah.org` was chosen as the new source (a long-stable, actively-maintained contribution-heatmap SVG generator) instead of trying to resurrect or self-host `github-readme-activity-graph`. `.github/workflows/stats.yml` now has a `curl --fail` step that fetches `assets/activity-graph.svg` alongside the existing top-langs step; `--fail` makes the step error loudly (rather than silently committing an error page) if this source ever goes down too.

In [ ]:
%%bash
# Evidence: the old activity-graph source is dead, not flaky.
curl -sL --max-time 15 -o /dev/null -w "old source (github-readme-activity-graph.vercel.app): HTTP %{http_code}\n" \
  "https://github-readme-activity-graph.vercel.app/graph?username=josejorge&theme=tokyo-night&hide_border=true"

# Evidence: the replacement source is alive and returns a real SVG.
curl -sL --max-time 15 -o /dev/null -w "new source (ghchart.rshah.org): HTTP %{http_code}\n" \
  "https://ghchart.rshah.org/0e75b6/josejorge"

## 2026-09-14 — Merging 3 blogs into "Latest Blog Posts" required treating all 3 feeds as equally Cloudflare-protected

**Context:** `scripts/update-blog-readme.js` already fetched `blog.kythex.com/feed` through a real headless browser (Playwright) because Cloudflare's Bot Fight Mode issues a silent JS challenge to plain HTTP clients from datacenter ASNs — which is exactly what GitHub-hosted Actions runners are. The task was to add `thealzdiary.com/feed` and `tierradeoz.com/feed`.

**Gotcha avoided:** a plain `curl` from a residential/local IP against both new feeds returned `200` with clean RSS XML directly — no visible Cloudflare challenge. It would have been tempting to conclude these two feeds don't need the browser workaround and fetch them with a lighter-weight plain HTTP request in CI.

**Why that would have been wrong:** Cloudflare Bot Fight Mode's challenge targets requests from datacenter/hosting-provider ASNs specifically — a local residential-IP test passing proves nothing about how the *same* request will behave from a GitHub Actions runner's IP range. Since the failure mode (silent challenge page mistaken for the real feed) is hard to detect after the fact, all three feeds are now fetched through the same shared Playwright browser/context in `fetchAllBlogs()`, uniformly, rather than special-casing two of them as "safe" based on a non-representative local test.

**Output shape decision:** the user asked for a 3-column Markdown table (one column per blog) instead of a single interleaved-by-date list, so each blog gets its own column with up to 5 of its most recent posts; a blog with fewer recent posts than another just leaves blank cells in its column rather than another blog's post bleeding into that row.

## 2026-09-14 — "Featured Work" section added, Jireh case study kept anonymized

**Context:** the README led entirely with a KytheX/hacker-terminal/gaming persona before any real engineering proof, which is weak for recruiters skimming the profile. A "Featured Work" section was added near the top, right after the About Me block.

**Decision on scope:** two projects were included — `calendar-appointments` (public repo under the `josejorge` GitHub account, linked directly) and a payroll + notification platform built for a client. The client project's actual repos (`apharenxis/jirehhomecleaning*`) are private and under a different account — not the user's to link publicly. Per explicit instruction, that case study is described generically ("a home-services company", no client name, no repo link) rather than naming the client, even though the client's own public-facing site exists — confidentiality/scope was the deciding factor, not technical constraint.